# M7.3 · Mixed precision: fp16, bf16, AMP

Düşük hassasiyetle eğitmenin tuzakları. fp16 taşmasını, bf16'nın hassasiyetini ve gradyan alt taşmasını elle hesaplıyor, saf Python'da herhangi bir kayan nokta biçimine yuvarlama ve dinamik loss scaler yazıyor, NumPy'da fp16 ağırlıklarla eğitimin neden takıldığını ve fp32 master kopyanın bunu nasıl çözdüğünü görüyor, PyTorch'ta `torch.autocast` + `GradScaler` ile eğitim adımı kuruyor ve en sonda saf bf16 eğitimde stochastic rounding'i bit düzeyinde vektörel olarak uyguluyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-3-amp.html](https://m-gonul.github.io/pytorch-lab/m7-3-amp.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_3_amp.py
CHECKS = {
 "b": "import math\nimport numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef bf16_ref(x):\n    u = np.array([x], dtype=np.float32).view(np.uint32).astype(np.uint64)\n    if not np.isfinite(np.float32(x)):\n        return float(np.float32(x))\n    u = (u + 0x7FFF + ((u >> 16) & 1)) & 0xFFFF0000\n    return float(u.astype(np.uint32).view(np.float32)[0])\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rf = need(ns, \"round_float\")\n\n    def f16():\n        rng = np.random.default_rng(0)\n        xs = np.concatenate([rng.normal(size=300) * 10.0 ** rng.integers(-9, 6, 300), [65504, 65519, 65520, 7e4, 2 ** -24, 2 ** -25, 3 * 2 ** -26, 1e-8, -0.1, 1 + 2 ** -11, 1 + 3 * 2 ** -11]])\n        for x in xs:\n            with np.errstate(over=\"ignore\"):\n                want = float(np.float16(x))\n            got = rf(float(x), 10, 5)\n            ensure(got == want or (math.isinf(got) and math.isinf(want) and (got > 0) == (want > 0)), f\"fp16({x!r}) = {got}, NumPy {want}\")\n    R.test(\"fp16 = NumPy float16 (taşma, subnormal, eşitlikte çift dahil)\", f16)\n\n    def b16():\n        rng = np.random.default_rng(1)\n        xs = np.float32(rng.normal(size=300) * 10.0 ** rng.integers(-30, 30, 300))\n        for x in list(xs) + [np.float32(1 + 2 ** -8), np.float32(1 + 3 * 2 ** -8)]:\n            ensure(rf(float(x), 7, 8) == bf16_ref(float(x)), f\"bf16({float(x)!r}) = {rf(float(x), 7, 8)}, beklenen {bf16_ref(float(x))}\")\n    R.test(\"bf16 = bit düzeyinde yuvarlama\", b16)\n\n    def ls():\n        S = need(ns, \"LossScaler\", \"sınıf\")(8.0)\n        seq = []\n        for inf in [False, False, False, True, False, False, False, False]:\n            S.update(inf); seq.append(S.scale)\n        ensure(seq == [8, 8, 16, 8, 8, 8, 16, 16], f\"ölçek dizisi {seq}\")\n    R.test(\"LossScaler: büyüme ve geri çekilme\", ls)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"train_fp16\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    X = rng.normal(size=(256, 10)).astype(np.float32)\n    w_true = (1 + 0.003 * np.arange(10)).astype(np.float32)\n    y = X @ w_true\n\n    def stall():\n        naive = need(ns, \"train_fp16\")(X, y, 0.01, 300, master=False)\n        mast = ns[\"train_fp16\"](X, y, 0.01, 300, master=True)\n        start = float(np.mean((X @ np.ones(10, np.float32) - y) ** 2))\n        ensure(naive > 0.3 * start, f\"fp16 ağırlıklar takılmalıydı: başlangıç {start:.2e}, son {naive:.2e}\")\n        ensure(mast < 0.05 * naive, f\"master kopya kaybı düşürmeli: {mast:.2e} vs {naive:.2e}\")\n        R.table(\"fp16 ağırlıklarla 300 GD adımı\", [\"\", \"kayıp\"],\n                [[\"başlangıç\", f\"{start:.2e}\"], [\"fp16 ağırlık\", f\"{naive:.2e}\"], [\"fp32 master + fp16 hesap\", f\"{mast:.2e}\"]],\n                note=\"Güncellemeler (~1e-5) ağırlığın yarım ulp'sinden (~4.9e-4) küçük: fp16'da kayboluyor.\")\n    R.test(\"fp16 ağırlıklar takılıyor, master kopya çözüyor\", stall)\n\n    def sg():\n        a, b = need(ns, \"scaled_grad\")(1e-8, 65536.0)\n        ensure(float(a) == 0.0, \"ölçeksiz 1e-8 fp16'da 0 olmalı\")\n        ensure(abs(float(b) - 1e-8) / 1e-8 < 1e-3, f\"ölçekli geri dönüş {float(b):.3e}\")\n    R.test(\"loss scaling alt taşmayı önlüyor\", sg)\n",
 "d": "import copy\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    dt = torch.float16 if dev == \"cuda\" else torch.bfloat16\n    torch.manual_seed(0)\n    model = torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Linear(64, 3)).to(dev)\n    x, y = torch.randn(256, 20, device=dev), torch.randint(0, 3, (256,), device=dev)\n\n    def dts():\n        d = need(ns, \"amp_dtypes\")(model, x, y, dev, dt)\n        ensure(d[\"linear\"] == dt, f\"Linear çıktısı {d['linear']}; autocast altında {dt} bekleniyordu\")\n        ensure(d[\"loss\"] == torch.float32, f\"cross_entropy fp32'de hesaplanmalı ({d['loss']})\")\n        ensure(d[\"param\"] == torch.float32 and d[\"grad\"] == torch.float32, \"parametre ve gradyan fp32 kalmalı\")\n        return \", \".join(f\"{k}: {str(v).replace('torch.', '')}\" for k, v in d.items())\n    R.test(\"autocast dtype'ları\", dts)\n\n    def train():\n        m = copy.deepcopy(model)\n        opt = torch.optim.Adam(m.parameters(), lr=1e-2)\n        scaler = torch.amp.GradScaler(dev)\n        first = need(ns, \"train_step_amp\")(m, opt, scaler, x, y, dev, dt)\n        for _ in range(150):\n            last = ns[\"train_step_amp\"](m, opt, scaler, x, y, dev, dt)\n        ensure(last < 0.5 * first, f\"kayıp {first:.3f} → {last:.3f}\")\n        return f\"kayıp {first:.3f} → {last:.3f}\"\n    R.test(\"AMP ile eğitim\", train)\n\n    def skip():\n        m = copy.deepcopy(model)\n        opt = torch.optim.SGD(m.parameters(), lr=0.1)\n        scaler = torch.amp.GradScaler(dev, init_scale=1024.0)\n        before = [p.detach().clone() for p in m.parameters()]\n        h = m[0].weight.register_hook(lambda g: g * float(\"inf\"))\n        need(ns, \"train_step_amp\")(m, opt, scaler, x, y, dev, dt)\n        h.remove()\n        for p, q in zip(m.parameters(), before):\n            close(p, q, what=\"inf gradyanlı adım atlanmalı\")\n        ensure(scaler.get_scale() < 1024.0, f\"ölçek küçülmeliydi ({scaler.get_scale()})\")\n    R.test(\"inf gradyan: adım atlanıyor, ölçek küçülüyor\", skip)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    g = torch.Generator().manual_seed(0)\n\n    def rep():\n        x = torch.randn(10000, generator=g) * 100\n        r = need(ns, \"sr_bf16\")(x, g)\n        ensure(r.dtype == torch.float32, \"fp32 tensör döndürmeli\")\n        ensure(torch.equal(r, r.to(torch.bfloat16).float()), \"sonuç bf16'da tam temsil edilebilir olmalı\")\n        lo = x.to(torch.bfloat16).float()\n        ensure(((r - x).abs() < 2 * (x.abs() * 2 ** -7 + 1e-30)).all(), \"her değer komşu iki bf16 değerinden birine gitmeli\")\n    R.test(\"sonuçlar bf16'da temsil edilebilir\", rep)\n\n    def unbiased():\n        for v in (1.001, -3.3333, 1e-3 + 1e-6):\n            x = torch.full((200000,), v)\n            m = need(ns, \"sr_bf16\")(x, g).double().mean().item()\n            near = x.to(torch.bfloat16).double()[0].item()\n            ensure(abs(m - v) < 0.1 * abs(near - v) + 1e-6 * abs(v), f\"{v}: SR ortalaması {m}, en yakın {near}\")\n    R.test(\"SR yansız: E[SR(x)] ≈ x\", unbiased)\n\n    def train():\n        X = torch.randn(512, 10, generator=g)\n        wt = 1 + 0.003 * torch.arange(10.0)\n        y = X @ wt\n        start = ((X @ torch.ones(10) - y) ** 2).mean().item()\n        nearest = need(ns, \"train_bf16\")(X, y, 0.01, 400, \"nearest\")\n        stoch = ns[\"train_bf16\"](X, y, 0.01, 400, \"stochastic\", torch.Generator().manual_seed(1))\n        ensure(stoch < 0.2 * nearest, f\"SR {stoch:.2e}, en yakın {nearest:.2e}\")\n        R.table(\"Saf bf16 ağırlıkla 400 GD adımı\", [\"yuvarlama\", \"son kayıp\"],\n                [[\"başlangıç\", f\"{start:.2e}\"], [\"en yakına\", f\"{nearest:.2e}\"], [\"stokastik\", f\"{stoch:.2e}\"]],\n                note=\"En yakına yuvarlamada küçük güncellemeler sürekli aşağı yuvarlanıp kayboluyor; SR'de beklenen değerde korunuyor.\")\n    R.test(\"SR'li bf16 eğitim takılmıyor\", train)\n"
}

In [ ]:
import importlib, labcheck, checks_m7_3_amp as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir kayan nokta sayısı $(-1)^s\cdot1.m\cdot2^{e-\text{bias}}$:

| biçim | üs | mantis | en büyük | 1'in üstündeki adım |
|---|---|---|---|---|
| fp32 | 8 | 23 | $3.4\cdot10^{38}$ | $1.2\cdot10^{-7}$ |
| fp16 | 5 | 10 | 65504 | $9.8\cdot10^{-4}$ |
| bf16 | 8 | 7 | $3.4\cdot10^{38}$ | $7.8\cdot10^{-3}$ |

Düşük hassasiyetin üç sorunu:

1. **Taşma:** fp16'da $300^2=\infty$.
2. **Alt taşma:** fp16'da $10^{-8}=0$; küçük gradyanlar kaybolur → **loss scaling**: kaybı $S$ ile çarp, gradyanları $S$'ye böl.
3. **Kaybolan güncellemeler:** $w=1$ iken $w+10^{-4}=1$ (fp16). Çözüm: ağırlıkların **fp32 master** kopyası; ya da saf bf16 eğitimde **stochastic rounding**.

**AMP** (`torch.autocast`): matmul/conv gibi işlemler düşük hassasiyette, kayıp/normalizasyon/indirgeme gibi hassas işlemler fp32'de çalışır. Parametreler ve gradyanları fp32 kalır.

## 1 · Elle

<!-- çözüm -->

fp16'da $[1,2)$ aralığında ardışık sayılar $2^{-10}$ arayla dizilir. $10^{-4}$ bunun yarısından küçük, yani en yakına yuvarlama $1$'i geri verir. Bu bir "küçük hata" değil: güncelleme **tamamen** kaybolur ve eğitim durur.

Loss scaling'de pratik değerler $2^{10}$–$2^{16}$; ölçek dinamik olarak ayarlanır.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**fp16: 5 bit üs, 10 bit mantis; bf16: 8 bit üs, 7 bit mantis**

- `f16max`: fp16'nın en büyük sonlu değeri
- `of`: $300^2$ fp16'da taşar mı? (evet 1, hayır 0)
- `eps16`: fp16'da 1'in hemen üstündeki sayı ile 1'in farkı
- `eps16b`: bf16'da aynı fark

**Gradyanlar ve güncellemeler**

- `upd`: fp16 ağırlık $w=1$, güncelleme $10^{-4}$: $w+10^{-4}$ fp16'da kaç olur?
- `scale`: $10^{-8}$'lik bir gradyanı fp16'nın en küçük pozitif değerinin ($2^{-24}\approx5.96\cdot10^{-8}$) üstüne çıkaran en küçük 2'nin kuvveti ölçek

In [ ]:
cevaplar = dict(
    f16max=None,
    of=None,
    eps16=None,
    eps16b=None,
    upd=None,
    scale=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"f16max":[65504,0],"of":[1,0],"eps16":[0.0009765625,0.000001],"eps16b":[0.0078125,0],"upd":[1,0],"scale":[8,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `round_float(x, man_bits, exp_bits)` — IEEE tarzı biçime en yakına yuvarla (eşitlikte çift: Python `round`). bias $=2^{e-1}-1$; üs $E=\lfloor\log_2|x|\rfloor$ (`math.frexp`), alt sınır $1-\text{bias}$ (subnormal); adım $q=2^{E-m}$; sonuç $\text{round}(|x|/q)\cdot q$; en büyük sonlu değeri aşarsa $\pm\infty$. `0`, `inf`, `nan` olduğu gibi döner.
- `LossScaler(init=2**16, growth=2.0, backoff=0.5, interval=3)` — `update(found_inf)`: inf varsa ölçeği `backoff` ile çarp ve sayacı sıfırla; yoksa sayacı artır, `interval` ardışık iyi adımda ölçeği `growth` ile çarp. `scale` özniteliği.

Kontrol fp16'yı NumPy'ın `float16`'sıyla, bf16'yı bit düzeyinde yuvarlamayla karşılaştırır.

In [ ]:
%%writefile stage_b.py
import math


def round_float(x, man_bits, exp_bits):
    ...


class LossScaler:
    def __init__(self, init=2.0 ** 16, growth=2.0, backoff=0.5, interval=3):
        ...

    def update(self, found_inf):
        ...


print(round_float(300.0 ** 2, 10, 5), round_float(1 + 1e-4, 10, 5), round_float(1e-8, 10, 5))   # inf 1.0 0.0
s = LossScaler(8.0)
for inf in [False, False, False, True, False]:
    s.update(inf)
    print(s.scale)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; doğrusal regresyon $y=Xw^*$, tam batch GD. **Yalnız adım döngüsü.**

- `train_fp16(X, y, lr, steps, master)` → son kayıp (fp32'de hesaplanmış). Ağırlıklar `np.float16`; ileri geçiş ve gradyan fp16'da.
  - `master=False`: `w = (w - lr * g).astype(np.float16)`.
  - `master=True`: fp32 bir `w32` tut, `w32 -= lr * g.astype(np.float32)`, ileri geçiş için `w = w32.astype(np.float16)`.
- `scaled_grad(g, scale)` → `(np.float16(g), np.float16(g * scale) / scale)` (ikincisi fp32'de bölünür): alt taşmanın ölçekle önlenmesi.

In [ ]:
%%writefile stage_c.py
import numpy as np


def train_fp16(X, y, lr, steps, master):
    ...


def scaled_grad(g, scale):
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(256, 10)).astype(np.float32)
w_true = (1 + 0.003 * np.arange(10)).astype(np.float32)
y = X @ w_true
print(train_fp16(X, y, 0.01, 300, master=False), train_fp16(X, y, 0.01, 300, master=True))
print(scaled_grad(1e-8, 65536.0))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

CPU'da `torch.autocast("cpu", dtype=torch.bfloat16)`, GPU'da `("cuda", torch.float16)` + `GradScaler`.

- `amp_dtypes(model, x, y, device_type, dtype)` → `{"linear": …, "loss": …, "param": …, "grad": …}`: autocast içinde `model(x)` çıktısının ve `F.cross_entropy` kaybının dtype'ı, geri geçişten sonra ilk parametrenin ve gradyanının dtype'ı.
- `train_step_amp(model, opt, scaler, x, y, device_type, dtype)` → kayıp (float): autocast'li ileri geçiş, `scaler.scale(loss).backward()`, `scaler.step(opt)`, `scaler.update()`, `opt.zero_grad()`.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def amp_dtypes(model, x, y, device_type, dtype):
    ...


def train_step_amp(model, opt, scaler, x, y, device_type, dtype):
    ...


dev = "cuda" if torch.cuda.is_available() else "cpu"
dt = torch.float16 if dev == "cuda" else torch.bfloat16
torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Linear(64, 3)).to(dev)
x, y = torch.randn(32, 20, device=dev), torch.randint(0, 3, (32,), device=dev)
print(amp_dtypes(model, x, y, dev, dt))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Saf bf16 eğitim (master kopya yok) ve stochastic rounding.

- `sr_bf16(x, generator)` — fp32 tensörü bf16'ya **stokastik** yuvarla, sonucu fp32 olarak döndür. Bit hilesi: `x.view(torch.int32)`'ye $[0,2^{16})$'dan rastgele bir tamsayı ekle, alt 16 biti sıfırla (`& -65536`), `view(torch.float32)`. Döngü yok.
- `train_bf16(X, y, lr, steps, rounding, generator=None)` → son kayıp; ağırlıklar her güncellemeden sonra bf16'ya yuvarlanır: `"nearest"` → `w.to(torch.bfloat16).float()`, `"stochastic"` → `sr_bf16`.

Kontrol: SR yansız mı, sonuçlar bf16'da temsil edilebilir mi, ve SR'li eğitim en yakına yuvarlamalı eğitimden ne kadar iyi.

In [ ]:
%%writefile stage_e.py
import torch


def sr_bf16(x, generator=None):
    """fp32 → bf16'ya stokastik yuvarlanmış fp32. Döngü yok."""
    ...


def train_bf16(X, y, lr, steps, rounding, generator=None):
    ...


g = torch.Generator().manual_seed(0)
v = torch.full((100000,), 1.001)
print(sr_bf16(v, g).mean(), v.to(torch.bfloat16).float()[0])   # ≈ 1.001 vs 1.0

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Micikevicius ve ark. (2018): karışık hassasiyetli eğitim

[Micikevicius ve ark. — Mixed Precision Training (ICLR 2018)](https://arxiv.org/abs/1710.03740)

Makalenin üç tekniğini ayrı ayrı uygulayıp ablasyon yap: (1) fp32 ana ağırlık kopyası, (2) kayıp ölçekleme (sabit ve dinamik), (3) fp32 birikimli indirgemeler. Bir CNN ya da küçük Transformer'da gradyan değerlerinin histogramını (Şekil 3) çiz ve fp16'nın temsil edilebilir aralığının dışına düşenlerin oranını ölç; ölçeksiz fp16 eğitiminin ıraksadığını, tekniklerle fp32 doğruluğuna ulaşıldığını göster. bf16 ile karşılaştır.


**Adımlar**

1. Gradyan histogramını log2 kovalarla ve fp16 alt sınırıyla (2⁻²⁴) birlikte çiz.

**Kabul ölçütleri**

- Kayıp ölçeklemesiz fp16'da sıfıra düşen gradyan oranı yüksek.
- Tekniklerle fp32'ye eşit doğruluk.

## Kaynaklar

- [Micikevicius ve ark. — Mixed Precision Training](https://arxiv.org/abs/1710.03740) — arXiv
- [PyTorch Docs — Automatic Mixed Precision (torch.amp)](https://docs.pytorch.org/docs/stable/amp.html) — BSD-3-Clause
- [Kalamkar ve ark. — A Study of BFLOAT16 for Deep Learning Training](https://arxiv.org/abs/1905.12322) — arXiv
- [Gupta ve ark. — Deep Learning with Limited Numerical Precision (stochastic rounding)](https://arxiv.org/abs/1502.02551) — arXiv
- [fast.ai Part 2 — Mixed precision ders notları](https://course.fast.ai/Lessons/part2.html) — course22p2 kodu Apache-2.0